# KMeans Clustering Practical — Simple Exam Notebook

**Scenario:** Iris Flower Clustering  
**Dataset:** `iris.csv`  
**Ground Truth Target:** `target` (0 = Setosa, 1 = Versicolour, 2 = Virginica)

## What we will do
1. Load and inspect the dataset.
2. Scale features using `StandardScaler`.
3. Find the optimal number of clusters using the **Elbow Method** (WCSS / `inertia_`).
4. Fit KMeans with the optimal $K = 3$.
5. Evaluate clustering using metrics from `cheet.pdf`:
   - `silhouette_score`
   - `davies_bouldin_score`
   - `calinski_harabasz_score`
   - `adjusted_rand_score` (compared to true species)
6. Compare KMeans **with PCA vs without PCA**.
7. Visualize clusters and cluster centers using 2D scatter plots.

**Important:** Keep `iris.csv` in the same directory as this notebook.


## 1. Import the tools

All tools directly from the cheatsheet: pandas, numpy, matplotlib, seaborn, StandardScaler, PCA, KMeans, and clustering evaluation metrics.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score
)


## 2. Load the dataset and inspect it

- `head()` shows the first rows.
- `shape` shows rows and columns.
- `info()` shows data types.
- `isnull().sum()` checks missing values.


In [ ]:
df = pd.read_csv("iris.csv")

print("First five rows:")
display(df.head())

print("Dataset shape:", df.shape)
print("\nColumn information:")
df.info()

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nSummary statistics:")
display(df.describe())


## 3. Separate features (X) and ground truth (y)

We scale the 4 continuous measurements using `StandardScaler`.


In [ ]:
target_col = "target"

X = df.drop(columns=[target_col])
y_true = df[target_col]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled features shape:", X_scaled.shape)


## 4. Elbow Method to find optimal K (using WCSS / inertia_)

We test $K$ from 1 to 10 and plot `inertia_` to identify the elbow point.


In [ ]:
wcss = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(k_range, wcss, marker='o', linestyle='--')
plt.title("Elbow Method For Optimal K")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS (Inertia)")
plt.grid(True, alpha=0.3)
plt.show()


## 5. Fit KMeans without PCA (K = 3)

We fit KMeans with $K = 3$ and compute the clustering scores.


In [ ]:
kmeans_no_pca = KMeans(n_clusters=3, random_state=42)
kmeans_no_pca.fit(X_scaled)

labels_no_pca = kmeans_no_pca.labels_
centers_no_pca = kmeans_no_pca.cluster_centers_

print("WCSS (Inertia):", round(kmeans_no_pca.inertia_, 4))
print("Silhouette Score:", round(silhouette_score(X_scaled, labels_no_pca), 4))
print("Davies-Bouldin Score:", round(davies_bouldin_score(X_scaled, labels_no_pca), 4))
print("Calinski-Harabasz Score:", round(calinski_harabasz_score(X_scaled, labels_no_pca), 4))
print("Adjusted Rand Index (vs True Labels):", round(adjusted_rand_score(y_true, labels_no_pca), 4))


## 6. Fit KMeans with PCA (2 Principal Components)

We project features to 2 principal components using `PCA(n_components=2)` and fit KMeans.


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("Explained variance ratio by PC1 and PC2:", np.round(pca.explained_variance_ratio_, 4))
print("Total variance explained:", round(sum(pca.explained_variance_ratio_) * 100, 2), "%")

kmeans_pca = KMeans(n_clusters=3, random_state=42)
kmeans_pca.fit(X_pca)

labels_pca = kmeans_pca.labels_
centers_pca = kmeans_pca.cluster_centers_

print("\nWith PCA Metrics:")
print("Silhouette Score:", round(silhouette_score(X_pca, labels_pca), 4))
print("Davies-Bouldin Score:", round(davies_bouldin_score(X_pca, labels_pca), 4))
print("Calinski-Harabasz Score:", round(calinski_harabasz_score(X_pca, labels_pca), 4))
print("Adjusted Rand Index (vs True Labels):", round(adjusted_rand_score(y_true, labels_pca), 4))


## 7. Compare Models (With PCA vs Without PCA)


In [ ]:
comparison_df = pd.DataFrame({
    "Metric": ["Silhouette Score", "Davies-Bouldin Score", "Calinski-Harabasz Score", "Adjusted Rand Index"],
    "Without PCA": [
        silhouette_score(X_scaled, labels_no_pca),
        davies_bouldin_score(X_scaled, labels_no_pca),
        calinski_harabasz_score(X_scaled, labels_no_pca),
        adjusted_rand_score(y_true, labels_no_pca)
    ],
    "With PCA (2 PCs)": [
        silhouette_score(X_pca, labels_pca),
        davies_bouldin_score(X_pca, labels_pca),
        calinski_harabasz_score(X_pca, labels_pca),
        adjusted_rand_score(y_true, labels_pca)
    ]
})

display(comparison_df.round(4))


## 8. Visualize Clusters and Centers (2D Scatter Plot)

Cheatsheet visualization using `sns.scatterplot` and `plt.scatter` for cluster centers.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: KMeans Predicted Clusters on 2D PCA projection
sns.scatterplot(
    x=X_pca[:, 0], y=X_pca[:, 1],
    hue=labels_pca, palette="Set1",
    s=60, ax=axes[0]
)
axes[0].scatter(
    centers_pca[:, 0], centers_pca[:, 1],
    c='black', marker='X', s=200, label='Centroids'
)
axes[0].set_title("KMeans Clusters (PCA Components)")
axes[0].set_xlabel("PC 1")
axes[0].set_ylabel("PC 2")
axes[0].legend()

# Plot 2: Actual True Species
sns.scatterplot(
    x=X_pca[:, 0], y=X_pca[:, 1],
    hue=y_true, palette="Set2",
    s=60, ax=axes[1]
)
axes[1].set_title("True Species Distribution")
axes[1].set_xlabel("PC 1")
axes[1].set_ylabel("PC 2")

plt.tight_layout()
plt.show()
